In [1]:
import torch

print("PyTorch:", torch.__version__)

PyTorch: 2.10.0


# Reproducible token sampling on CPU and MPS

- When we generate text with an LLM, we often use `torch.softmax` to convert logits to probabilities and `torch.multinomial` to sample the next token.
- Unfortunately, it sometimes happens that CPU and MPS can sample different tokens from the same probabilities even when we use an identical random seed. That's likely because each device uses its own random number generator.
- In this notebook, I use a small example to show this behavior and make the sampling step consistent by running it on CPU for both cases.
- By the way, PyTorch does not guarantee identical results across devices or releases. See its [reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html) for details.
- The examples are intentionally minimal. For a complete LLM text generation function with temperature scaling, see [chapter 5](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/ch05.ipynb).

&nbsp;
## 1) Sampling on each device

- The following code hardcodes the random seed to `123` before each call so that we can compare CPU and MPS using the same seed.

In [2]:
logits = torch.tensor(
    [[0.0, 1.0, 2.0]],
    dtype=torch.bfloat16,
)

# Convert logits to probabilities
probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

def sample(probs):

    torch.manual_seed(123)

    # Sample from the distribution
    idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

    return idx_next


print("CPU:", sample(probs.to("cpu")))

if torch.mps.is_available():
    print("MPS:", sample(probs.to("mps")))

if torch.cuda.is_available():
    print("CUDA:", sample(probs.to("cuda")))

CPU: tensor([[1]])
MPS: tensor([[2]], device='mps:0')


&nbsp;
## 2) Sampling on CPU for both devices

- So one fix or workaround for the CPU/MPS discrepancy issue above is calling `probs.cpu()` before `torch.multinomial`.

In [3]:
def sample_on_cpu(probs):

    torch.manual_seed(123)

    # Sample from the distribution
    idx_next = torch.multinomial(probs.cpu(), num_samples=1)  # (batch_size, 1)

    return idx_next.to(probs.device)


print("CPU:", sample_on_cpu(probs.to("cpu")))

if torch.mps.is_available():
    print("MPS:", sample_on_cpu(probs.to("mps")))

if torch.cuda.is_available():
    print("CUDA:", sample_on_cpu(probs.to("cuda")))

CPU: tensor([[1]])
MPS: tensor([[1]], device='mps:0')


- By the way, this is only one aspect why results can differ on MPS and CPU.
- There can be other culprits, like a full LLM, CPU and MPS may produce slightly different logits. These differences can change the probabilities and sampled tokens, so generated text may still differ when sampling runs on CPU.
- Also, some issues are due to bugs in some chips, like M1 and M2, as desribed in the excellent [Finding a Bug in Closed-Source Shader Kernels](https://scottkirila.studio.site/blog/kernel-bug) investigation by Scott Kirila